# Baseline detection model

## Импорты, пути и воспроизводимость

In [1]:
from __future__ import annotations

import sys
import random
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
)
from torchmetrics.detection.mean_ap import MeanAveragePrecision

PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))

from src.data.paths import DATA_PROCESSED_DIR

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_PROCESSED_DIR:", DATA_PROCESSED_DIR)
print("DEVICE:", DEVICE)

PROJECT_DIR: c:\Users\maxim\OneDrive\Документы\aie_course_rep\aie_course_rep\project
DATA_PROCESSED_DIR: C:\Users\maxim\OneDrive\Документы\aie_course_rep\aie_course_rep\project\data\processed
DEVICE: cpu


## Конфигурация baseline

In [2]:
ANNOTATIONS_CSV = DATA_PROCESSED_DIR / "detection" / "annotations.csv"
IMAGES_DIR = DATA_PROCESSED_DIR / "detection" / "images"

BATCH_SIZE = 4
VAL_SPLIT = 0.2
NUM_WORKERS = 0

SCORE_THRESHOLD = 0.3
IOU_THRESHOLD = 0.5

# Baseline не обучается, но inference на всём val может быть долгим.
MAX_VAL_IMAGES = 300

BASELINE_DIR = PROJECT_DIR / "artifacts" / "baseline_models"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

print("ANNOTATIONS_CSV exists:", ANNOTATIONS_CSV.exists())
print("IMAGES_DIR exists:", IMAGES_DIR.exists())

ANNOTATIONS_CSV exists: True
IMAGES_DIR exists: True


## Загрузка и проверка аннотаций

In [3]:
ann_df = pd.read_csv(ANNOTATIONS_CSV)

required_columns = {
    "image",
    "label",
    "xmin",
    "ymin",
    "xmax",
    "ymax",
    "image_width",
    "image_height",
}

missing = required_columns - set(ann_df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

print("Annotations:", len(ann_df))
print("Images:", ann_df["image"].nunique())
print("Classes:", sorted(ann_df["label"].unique().tolist()))

display(ann_df.head())

Annotations: 194539
Images: 26300
Classes: ['biker', 'car', 'pedestrian', 'trafficLight', 'trafficLight-Green', 'trafficLight-GreenLeft', 'trafficLight-Red', 'trafficLight-RedLeft', 'trafficLight-Yellow', 'trafficLight-YellowLeft', 'truck']


,image,label,xmin,ymin,xmax,ymax,image_width,image_height
0,1478900859981702684_jpg.rf.6830635c7d919747563...,car,291,247,370,331,512,512
1,1478900859981702684_jpg.rf.6830635c7d919747563...,pedestrian,270,235,293,321,512,512
2,1478900859981702684_jpg.rf.6830635c7d919747563...,car,0,266,13,327,512,512
3,1478900859981702684_jpg.rf.6830635c7d919747563...,car,25,258,106,304,512,512
4,1478900859981702684_jpg.rf.6830635c7d919747563...,car,111,259,135,289,512,512


In [4]:
class_names = sorted(ann_df["label"].unique().tolist())

label_to_id = {label: idx + 1 for idx, label in enumerate(class_names)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print("label_to_id:")
print(label_to_id)

label_to_id:
{'biker': 1, 'car': 2, 'pedestrian': 3, 'trafficLight': 4, 'trafficLight-Green': 5, 'trafficLight-GreenLeft': 6, 'trafficLight-Red': 7, 'trafficLight-RedLeft': 8, 'trafficLight-Yellow': 9, 'trafficLight-YellowLeft': 10, 'truck': 11}


## Train/val split

In [5]:
def split_images_by_seed(
    image_names: list[str],
    val_split: float = 0.2,
    seed: int = 42,
) -> tuple[list[str], list[str]]:
    image_names = sorted(image_names)

    rng = random.Random(seed)
    rng.shuffle(image_names)

    val_size = int(len(image_names) * val_split)

    val_images = image_names[:val_size]
    train_images = image_names[val_size:]

    return train_images, val_images


all_images = ann_df["image"].unique().tolist()

train_images, val_images = split_images_by_seed(
    image_names=all_images,
    val_split=VAL_SPLIT,
    seed=SEED,
)

val_images_subset = val_images[:MAX_VAL_IMAGES]

train_df = ann_df[ann_df["image"].isin(train_images)].reset_index(drop=True)
val_df = ann_df[ann_df["image"].isin(val_images_subset)].reset_index(drop=True)

print("Train images:", len(train_images))
print("Val images full:", len(val_images))
print("Val images used for baseline:", len(val_images_subset))
print("Train annotations:", len(train_df))
print("Val annotations used:", len(val_df))

Train images: 21040
Val images full: 5260
Val images used for baseline: 300
Train annotations: 155519
Val annotations used: 2128


## Dataset для baseline-оценки

In [6]:
class DetectionDataset(Dataset):
    def __init__(
        self,
        images_dir: str | Path,
        annotations_df: pd.DataFrame,
        label_to_id: dict[str, int],
    ):
        self.images_dir = Path(images_dir)
        self.annotations_df = annotations_df.copy().reset_index(drop=True)
        self.label_to_id = label_to_id
        self.image_names = sorted(self.annotations_df["image"].unique().tolist())

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx: int):
        image_name = self.image_names[idx]
        rows = self.annotations_df[
            self.annotations_df["image"].eq(image_name)
        ].reset_index(drop=True)

        image_path = self.images_dir / image_name
        image = Image.open(image_path).convert("RGB")
        image_tensor = F.to_tensor(image)

        boxes = torch.tensor(
            rows[["xmin", "ymin", "xmax", "ymax"]].values,
            dtype=torch.float32,
        )

        labels = torch.tensor(
            [self.label_to_id[label] for label in rows["label"].tolist()],
            dtype=torch.int64,
        )

        widths = boxes[:, 2] - boxes[:, 0]
        heights = boxes[:, 3] - boxes[:, 1]
        valid = (widths > 1) & (heights > 1)

        boxes = boxes[valid]
        labels = labels[valid]

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64),
            "image_name": image_name,
        }

        return image_tensor, target

## Dataloaders

In [7]:
def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


val_dataset = DetectionDataset(
    images_dir=IMAGES_DIR,
    annotations_df=val_df,
    label_to_id=label_to_id,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

print("Val dataset:", len(val_dataset))
print("Val loader batches:", len(val_loader))

images, targets = next(iter(val_loader))
print("Batch images:", len(images))
print("First image shape:", images[0].shape)
print("First target keys:", targets[0].keys())

Val dataset: 300
Val loader batches: 75
Batch images: 4
First image shape: torch.Size([3, 512, 512])
First target keys: dict_keys(['boxes', 'labels', 'image_id', 'area', 'iscrowd', 'image_name'])


# Загрузка baseline модели

In [8]:
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT

baseline_model = fasterrcnn_resnet50_fpn(weights=weights)
baseline_model = baseline_model.to(DEVICE)
baseline_model.eval()

coco_categories = weights.meta["categories"]

print("Baseline model: fasterrcnn_resnet50_fpn")
print("Weights: COCO pretrained")
print("Fine-tuning on Udacity: no")
print("COCO categories:", coco_categories[:15])
print("Number of COCO categories:", len(coco_categories))

Baseline model: fasterrcnn_resnet50_fpn
Weights: COCO pretrained
Fine-tuning on Udacity: no
COCO categories: ['__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign', 'parking meter']
Number of COCO categories: 91


In [9]:
COCO_TO_UDACITY = {
    "car": "car",
    "truck": "truck",
    "person": "pedestrian",
    "traffic light": "trafficLight",
    "bicycle": "biker",
    "motorcycle": "biker",
}

coco_id_to_udacity_id = {}

for coco_id, coco_name in enumerate(coco_categories):
    if coco_name in COCO_TO_UDACITY:
        udacity_name = COCO_TO_UDACITY[coco_name]

        if udacity_name in label_to_id:
            coco_id_to_udacity_id[coco_id] = label_to_id[udacity_name]

print("COCO -> Udacity label mapping:")
print(coco_id_to_udacity_id)

COCO -> Udacity label mapping:
{1: 3, 2: 1, 3: 2, 4: 1, 8: 11, 10: 4}


## Wrapper для приведения предсказаний к формату проекта

In [10]:
class CocoToUdacityModelWrapper(torch.nn.Module):
    def __init__(
        self,
        model: torch.nn.Module,
        coco_id_to_udacity_id: dict[int, int],
    ):
        super().__init__()
        self.model = model
        self.coco_id_to_udacity_id = coco_id_to_udacity_id

    @torch.no_grad()
    def forward(self, images):
        outputs = self.model(images)

        mapped_outputs = []

        for output in outputs:
            boxes = output["boxes"]
            scores = output["scores"]
            labels = output["labels"]

            keep_indices = []
            mapped_labels = []

            for i, label in enumerate(labels):
                coco_id = int(label.item())

                if coco_id in self.coco_id_to_udacity_id:
                    keep_indices.append(i)
                    mapped_labels.append(self.coco_id_to_udacity_id[coco_id])

            if keep_indices:
                keep_indices = torch.tensor(
                    keep_indices,
                    dtype=torch.long,
                    device=boxes.device,
                )

                mapped_output = {
                    "boxes": boxes[keep_indices],
                    "scores": scores[keep_indices],
                    "labels": torch.tensor(
                        mapped_labels,
                        dtype=torch.int64,
                        device=boxes.device,
                    ),
                }
            else:
                mapped_output = {
                    "boxes": torch.empty((0, 4), dtype=torch.float32, device=boxes.device),
                    "scores": torch.empty((0,), dtype=torch.float32, device=boxes.device),
                    "labels": torch.empty((0,), dtype=torch.int64, device=boxes.device),
                }

            mapped_outputs.append(mapped_output)

        return mapped_outputs


baseline_model_mapped = CocoToUdacityModelWrapper(
    model=baseline_model,
    coco_id_to_udacity_id=coco_id_to_udacity_id,
).to(DEVICE)

baseline_model_mapped.eval()

CocoToUdacityModelWrapper(
  (model): FasterRCNN(
    (transform): GeneralizedRCNNTransform(
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        Resize(min_size=(800,), max_size=1333, mode='bilinear')
    )
    (backbone): BackboneWithFPN(
      (body): IntermediateLayerGetter(
        (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (bn1): FrozenBatchNorm2d(64, eps=0.0)
        (relu): ReLU(inplace=True)
        (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
        (layer1): Sequential(
          (0): Bottleneck(
            (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn1): FrozenBatchNorm2d(64, eps=0.0)
            (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (bn2): FrozenBatchNorm2d(64, eps=0.0)
            (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


## Расчет IoU и matching предсказаний

In [11]:
def box_iou_single(box_a: torch.Tensor, box_b: torch.Tensor) -> float:
    x_left = max(float(box_a[0]), float(box_b[0]))
    y_top = max(float(box_a[1]), float(box_b[1]))
    x_right = min(float(box_a[2]), float(box_b[2]))
    y_bottom = min(float(box_a[3]), float(box_b[3]))

    inter_w = max(0.0, x_right - x_left)
    inter_h = max(0.0, y_bottom - y_top)
    inter_area = inter_w * inter_h

    area_a = max(0.0, float(box_a[2] - box_a[0])) * max(
        0.0,
        float(box_a[3] - box_a[1]),
    )
    area_b = max(0.0, float(box_b[2] - box_b[0])) * max(
        0.0,
        float(box_b[3] - box_b[1]),
    )

    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0

    return inter_area / union


def match_predictions_to_targets(
    pred_boxes: torch.Tensor,
    pred_labels: torch.Tensor,
    pred_scores: torch.Tensor,
    true_boxes: torch.Tensor,
    true_labels: torch.Tensor,
    iou_threshold: float = 0.5,
) -> tuple[int, int, int]:
    if len(pred_boxes) == 0 and len(true_boxes) == 0:
        return 0, 0, 0

    if len(pred_boxes) == 0:
        return 0, 0, len(true_boxes)

    if len(true_boxes) == 0:
        return 0, len(pred_boxes), 0

    order = torch.argsort(pred_scores, descending=True)
    pred_boxes = pred_boxes[order]
    pred_labels = pred_labels[order]

    matched_gt = set()
    tp = 0
    fp = 0

    for pred_idx in range(len(pred_boxes)):
        best_iou = 0.0
        best_gt_idx = -1

        for gt_idx in range(len(true_boxes)):
            if gt_idx in matched_gt:
                continue

            if int(pred_labels[pred_idx]) != int(true_labels[gt_idx]):
                continue

            iou = box_iou_single(pred_boxes[pred_idx], true_boxes[gt_idx])

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_gt_idx >= 0 and best_iou >= iou_threshold:
            tp += 1
            matched_gt.add(best_gt_idx)
        else:
            fp += 1

    fn = len(true_boxes) - len(matched_gt)

    return tp, fp, fn

In [12]:
@torch.no_grad()
def evaluate_detection(
    model,
    dataloader,
    device: str | torch.device,
    score_threshold: float = 0.3,
    iou_threshold: float = 0.5,
) -> dict[str, float]:
    model.eval()

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for images, targets in dataloader:
        images = [img.to(device) for img in images]

        outputs = model(images)

        for output, target in zip(outputs, targets):
            pred_boxes = output["boxes"].detach().cpu()
            pred_labels = output["labels"].detach().cpu()
            pred_scores = output["scores"].detach().cpu()

            keep = pred_scores >= score_threshold

            pred_boxes = pred_boxes[keep]
            pred_labels = pred_labels[keep]
            pred_scores = pred_scores[keep]

            true_boxes = target["boxes"].detach().cpu()
            true_labels = target["labels"].detach().cpu()

            tp, fp, fn = match_predictions_to_targets(
                pred_boxes=pred_boxes,
                pred_labels=pred_labels,
                pred_scores=pred_scores,
                true_boxes=true_boxes,
                true_labels=true_labels,
                iou_threshold=iou_threshold,
            )

            total_tp += tp
            total_fp += fp
            total_fn += fn

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "precision@0.5": precision,
        "recall@0.5": recall,
        "f1@0.5": f1,
        "tp": total_tp,
        "fp": total_fp,
        "fn": total_fn,
    }


@torch.no_grad()
def evaluate_detection_map(
    model,
    dataloader,
    device: str | torch.device,
) -> dict[str, float]:
    model.eval()

    metric = MeanAveragePrecision()

    for images, targets in dataloader:
        images = [img.to(device) for img in images]

        outputs = model(images)

        preds_for_map = []
        targets_for_map = []

        for output, target in zip(outputs, targets):
            preds_for_map.append(
                {
                    "boxes": output["boxes"].detach().cpu(),
                    "scores": output["scores"].detach().cpu(),
                    "labels": output["labels"].detach().cpu(),
                }
            )

            targets_for_map.append(
                {
                    "boxes": target["boxes"].detach().cpu(),
                    "labels": target["labels"].detach().cpu(),
                }
            )

        metric.update(preds_for_map, targets_for_map)

    result = metric.compute()

    return {
        "map": float(result["map"]),
        "map@0.5": float(result["map_50"]),
        "map@0.75": float(result["map_75"]),
        "mar_1": float(result["mar_1"]),
        "mar_10": float(result["mar_10"]),
        "mar_100": float(result["mar_100"]),
    }


def run_full_detection_evaluation(
    model,
    dataloader,
    device: str | torch.device,
    score_threshold: float = 0.3,
    iou_threshold: float = 0.5,
) -> dict[str, float]:
    prf_metrics = evaluate_detection(
        model=model,
        dataloader=dataloader,
        device=device,
        score_threshold=score_threshold,
        iou_threshold=iou_threshold,
    )

    map_metrics = evaluate_detection_map(
        model=model,
        dataloader=dataloader,
        device=device,
    )

    return {
        **prf_metrics,
        **map_metrics,
    }

# Запуск baseline eval

In [13]:
baseline_metrics = run_full_detection_evaluation(
    model=baseline_model_mapped,
    dataloader=val_loader,
    device=DEVICE,
    score_threshold=SCORE_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
)

baseline_metrics

{'precision@0.5': 0.3562421185372005,
 'recall@0.5': 0.5310150375939849,
 'f1@0.5': 0.4264150943396226,
 'tp': 1130,
 'fp': 2042,
 'fn': 998,
 'map': 0.06456451863050461,
 'map@0.5': 0.13625763356685638,
 'map@0.75': 0.05333368852734566,
 'mar_1': 0.05899733677506447,
 'mar_10': 0.1274515688419342,
 'mar_100': 0.13217750191688538}

## Сохранение результатов 

In [14]:
baseline_payload = {
    "experiment_id": "BASE_FR50_COCO_NO_FINETUNE",
    "model_name": "fasterrcnn_resnet50_fpn",
    "weights": "COCO pretrained",
    "finetuned_on_udacity": False,
    "dataset": "Udacity Self Driving Car Dataset",
    "val_images_used": len(val_dataset),
    "score_threshold": SCORE_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "metrics": baseline_metrics,
    "coco_to_udacity_mapping": {
        str(k): v for k, v in coco_id_to_udacity_id.items()
    },
}

baseline_metrics_path = BASELINE_DIR / "pretrained_fasterrcnn_baseline_metrics.json"

with baseline_metrics_path.open("w", encoding="utf-8") as f:
    json.dump(baseline_payload, f, ensure_ascii=False, indent=2)

print("Saved:", baseline_metrics_path)

Saved: c:\Users\maxim\OneDrive\Документы\aie_course_rep\aie_course_rep\project\artifacts\baseline_models\pretrained_fasterrcnn_baseline_metrics.json


## Итоговый вывод

В ноутбуке была оценена baseline-модель `Faster R-CNN ResNet50 FPN` с COCO pretrained weights без дообучения на целевом датасете.

Baseline показал ограниченное качество, что ожидаемо для модели, не адаптированной к Udacity Self Driving Car Dataset. Полученные результаты подтверждают необходимость fine-tuning на целевых данных и используются как стартовая точка перед основными detection-экспериментами.